# Lesson 0.3: How a small model produces a reply

Load Qwen3.5-2B, look at the prompt it really sees, and write the generation loop yourself.

**Before running**
- Colab: Runtime → Change runtime type → **T4 GPU**.
- Kaggle: Settings → Accelerator **GPU T4**, Internet **on**.
- `HF_TOKEN` is optional here (Qwen3.5-2B is public), but it gives faster downloads.

The setup cells and the two lesson snippets are filled in. **The five exercises are yours**: each has an empty code cell and a notes cell.

## Setup

In [ ]:
# Qwen3.5 needs Transformers v5.
# If a later cell fails with an import error, use Runtime -> Restart session, then run from the next cell.
%pip install -q -U "transformers>=5" accelerate

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Kaggle has two T4s; use one so results match Colab

try:                                       # optional login for faster downloads
    try:
        from kaggle_secrets import UserSecretsClient
        get = UserSecretsClient().get_secret
    except ImportError:
        from google.colab import userdata
        get = userdata.get
    from huggingface_hub import login
    login(token=get("HF_TOKEN"))
    print("Logged in to Hugging Face")
except Exception:
    print("No HF_TOKEN found: downloading anonymously (fine for this lesson)")

## Snippet 1: load the model and look at the prompt it really sees

The model card loads Qwen3.5 with the multimodal classes; text-only messages work the same way.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

MID = "Qwen/Qwen3.5-2B"
proc = AutoProcessor.from_pretrained(MID)
tok = proc.tokenizer
model = AutoModelForMultimodalLM.from_pretrained(
    MID, device_map="auto", dtype=torch.float16)   # the T4 has no bf16

tools = [{"type": "function", "function": {
    "name": "get_order_status",
    "description": "Status of one of the signed-in customer's orders",
    "parameters": {"type": "object",
                   "properties": {"order_id": {"type": "integer"}},
                   "required": ["order_id"]}}}]

messages = [
    {"role": "system", "content": "You are Nova, NimbusCart's support assistant."},
    {"role": "user", "content": "Hi, where is my order 4417?"},
]

text = tok.apply_chat_template(messages, tools=tools,
                               add_generation_prompt=True, tokenize=False)
print(text)
print(len(tok(text).input_ids), "tokens")

## Snippet 2: the generation loop, by hand

No KV cache, so it is slow, and that is the point: you can see each step.
If the processor complains about plain-string content, wrap it as `[{"type": "text", "text": "..."}]`.

In [ ]:
ids = tok(text, return_tensors="pt").input_ids.to(model.device)
stop = tok.convert_tokens_to_ids("<|im_end|>")

for _ in range(80):
    with torch.no_grad():
        logits = model(input_ids=ids).logits[:, -1, :]
    nxt = logits.argmax(dim=-1, keepdim=True)        # greedy pick
    ids = torch.cat([ids, nxt], dim=-1)
    if nxt.item() == stop:
        break

print(tok.decode(ids[0, -80:]))

## Exercise 1: find every special token

Print the rendered prompt **with** and **without** `tools`. For each special token (`<|im_start|>`, `<|im_end|>` and any others), write what it marks.

Then work out how many tokens the tool schema costs (tokens with tools minus tokens without) and put that number in the context budget in `DESIGN.md`; it says 700 there as a guess.

In [ ]:
# Your code here

**My notes:**
- Special tokens and what they mark:
- Tokens without tools: ___  with tools: ___  so the tool schema costs: ___

## Exercise 2: reproduce the wall

Generate a reply three ways and compare:
1. From the raw string `"Hi, where is my order 4417?"` with **no template**.
2. From the template (`add_generation_prompt=True`).
3. From the template with `add_generation_prompt=False`.

Hint: you can reuse the loop from Snippet 2; only the starting `text` changes.

In [ ]:
# Your code here

**My notes:** what happened in each case, and why:
1.
2.
3.

## Exercise 3: does your loop match `generate`?

Run `model.generate(...)` with `do_sample=False` on the same input and compare its text with your loop's. They should match exactly. If they don't, find out why.

Hint: decode only the **new** tokens from each, and keep `skip_special_tokens=False` so nothing is hidden.

In [ ]:
# Your code here

**My notes:** did they match? If not, what was different?

## Exercise 4: how much does sampling change the answer?

Generate the same reply 5 times at `temperature=0.2`, and 5 times with the model card's non-thinking settings: `temperature=1.0, top_p=1.0, top_k=20` (plus `presence_penalty=2.0`).

Note how much the content and length vary.

Hint: sampling needs `do_sample=True`. If `generate()` rejects `presence_penalty`, leave it out: it is a serving (vLLM) setting you will use in Module 10.

In [ ]:
# Your code here

**My notes:** how much did content and length vary at each setting?

## Exercise 5: does the base model call the tool?

With `tools` provided, ask "Where is my order 4417?" and look at the **raw** output, special tokens included. Does the untrained model produce a tool call?

Save the raw output: copy it into `agent/fixtures/base_tool_call_raw.txt` in the repo on your laptop. In Module 6 you will write code that parses exactly this format.

In [ ]:
# Your code here

**My notes:** did it call the tool? What does the tool-call format look like?

## When you're done

- [ ] Qwen3.5-2B answers a chat-formatted prompt
- [ ] Your hand-written loop matches `generate`
- [ ] Tool-schema token cost written into `DESIGN.md`
- [ ] Raw tool-call output saved to `agent/fixtures/base_tool_call_raw.txt`